# Can we predict `passed_quiz`?
Protocol: `protocol.yaml` (group holdout on `student_key` from `holdout.csv`, 5-fold GroupKFold CV on training rows, choose by mean CV ROC AUC, score holdout once).

Feature timing (from `columns.csv`): the quiz closes the session, so prediction time is just before the quiz.
- `enrol_year`, `school_code`, `prior_score` (placement test): known beforehand -> used.
- `hints_used`: accrued during the session, before the closing quiz -> used.
- `session_minutes`: length of the whole session, which ends with the quiz, so it is only known after the outcome -> **excluded** as leakage.
- `student_key`: entity key -> excluded (per protocol).

In [ ]:
import json, hashlib, numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv('data.csv')
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='5a0b0221ed641db40322e1a95cae32cc67d3525ecb500813f95d6c109b2a7409'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='ff08c7a88fd7e15b17b05a390393550fdf57ea2b6df584f1d0594cc2ec1f00e6'
hold_rows = pd.read_csv('holdout.csv')['row'].to_numpy()
is_hold = np.zeros(len(df), bool); is_hold[hold_rows] = True
train, hold = df[~is_hold], df[is_hold]
assert len(hold)==2465 and not set(train.student_key)&set(hold.student_key)
print(len(train), len(hold), train.passed_quiz.mean(), hold.passed_quiz.mean())

In [ ]:
FEATURES = ['enrol_year','school_code','prior_score','hints_used']
CAT = ['school_code']; NUM = ['enrol_year','prior_score','hints_used']
def prep(cat=True):
    return ColumnTransformer([('num',StandardScaler(),NUM)] + ([('cat',OneHotEncoder(handle_unknown='ignore',min_frequency=20),CAT)] if cat else []))
# all candidates named before any scoring
candidates = {
 'baseline_prior': Pipeline([('m',DummyClassifier(strategy='prior'))]),
 'logreg_C1': Pipeline([('p',prep()),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'logreg_C0.1': Pipeline([('p',prep()),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_nocat': Pipeline([('p',prep(False)),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'hgb_depth3': Pipeline([('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
 'rf_leaf20': Pipeline([('m',RandomForestClassifier(n_estimators=300,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
}
print(list(candidates))

In [ ]:
X, y, g = train[FEATURES], train.passed_quiz.to_numpy(), train.student_key
folds = list(GroupKFold(n_splits=5).split(X,y,g))
cv_scores = {}
for name,est in candidates.items():
    s=[]
    for tr,va in folds:
        m=clone(est).fit(X.iloc[tr],y[tr])
        s.append(roc_auc_score(y[va],m.predict_proba(X.iloc[va])[:,1]))
    cv_scores[name]=s
    print(f'{name:16s} mean={np.mean(s):.4f} sd={np.std(s):.4f}')
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
print('chosen:', chosen)

In [ ]:
final = clone(candidates[chosen]).fit(X,y)
holdout_auc = roc_auc_score(hold.passed_quiz, final.predict_proba(hold[FEATURES])[:,1])
print('Holdout ROC AUC (scored once):', round(holdout_auc,4))

In [ ]:
train_rows = np.flatnonzero(~is_hold)
evidence = {
 'features': FEATURES,
 'candidates': list(candidates),
 'cv_folds': [[train_rows[tr].tolist(), train_rows[va].tolist()] for tr,va in folds],
 'cv_scores': {k:[float(v) for v in s] for k,s in cv_scores.items()},
 'chosen': chosen,
 'holdout_rows': hold_rows.tolist(),
 'holdout_scores': {chosen: float(holdout_auc)},
}
json.dump(evidence, open('evidence.json','w'))

## Conclusion
The headline is the holdout ROC AUC of the CV-chosen candidate (printed above), using only pre-quiz features. Excluding `session_minutes` (known only after the outcome) costs signal, but including it would overstate what is predictable at prediction time.